<a href="https://colab.research.google.com/github/mustafayubk/ps1-overleaf-template/blob/main/companion/notebooks/06_ps1_strategic_reasoning_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PS1: Observability and Sovereign Disclosure — Public Goods Game

COMSCI/ECON 206 · Mustafa Ayub Khan

This notebook implements the observability-based public goods game described in the PS1 proposal (Section 3). It reproduces the deployed Hugging Face demo's payoff arithmetic in Python, verifies it against hand-calculated predictions, and extends the single fixed-schedule demo into a broader sweep across many randomly drawn peer schedules.

Run all cells from a fresh CPU session. No LLM API call is used.

In [5]:
# Run this cell before importing scientific packages. CPU only; no API key.
import sys, subprocess, importlib.metadata as md
pins = {'numpy':'2.3.5', 'scipy':'1.17.0', 'matplotlib':'3.10.8', 'nashpy':'0.0.43'}
missing=[]
for package, version in pins.items():
    try: installed=md.version(package)
    except md.PackageNotFoundError: installed=None
    if installed != version: missing.append(f'{package}=={version}')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
print('Tested with Python 3.12; this runtime:',sys.version.split()[0])
print({p:md.version(p) for p in pins})
# If Colab asks to restart after installation, restart the session and Run all.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(7,3.6),'axes.spines.top':False,'axes.spines.right':False})


Tested with Python 3.12; this runtime: 3.13.15
{'numpy': '2.3.5', 'scipy': '1.17.0', 'matplotlib': '3.10.8', 'nashpy': '0.0.43'}


## Three connected questions

1. **Economics:** Does higher-quality sovereign climate-risk disclosure causally lower a country's borrowing cost, beyond what rating level alone explains?
2. **Computation:** Can a reproducible, observability-based public goods game show whether visibility of peers' prior contributions alone — with no payoff change — raises voluntary contribution?
3. **Behavioral science:** Does increased peer visibility raise genuine transparency, or does it teach actors to perform visible disclosure without substantively improving it ("disclosure theater")?

**Model.** Each round, a player splits an endowment of 10 tokens between a Personal Fund (paying 4 per token, to the contributor only) and a Group Fund (paying 2 per token, to all 5 players). Since 4 > 2, contributing 0 strictly dominates any positive contribution — the standard public-goods free-rider result. This matches the model in Section 2 of the proposal, with $a=4$, $b=2$, $N=5$.

In [6]:
"""Public goods game engine, matching the deployed Hugging Face demo."""
import numpy as np

A_PRIVATE = 4  # payoff per token kept in Personal Fund
B_GROUP = 2    # payoff per token in Group Fund, paid to each of 5 players
ENDOWMENT = 10

PEER_LOW = [10, 10, 10]     # rounds 1-3: low observability, sum = 10 each
PEER_HIGH = [10, 10, 10]    # rounds 4-6: high observability, sum = 10 each (matches deployed game)
PEER_SCHEDULE = PEER_LOW + PEER_HIGH

def round_payoff(c, peer_sum, a=A_PRIVATE, b=B_GROUP, e=ENDOWMENT):
    """Stage payoff to the player for contribution c, given peers' summed contribution."""
    pool = c + peer_sum
    return a * (e - c) + b * pool

def nash_best_response(peer_sum, a=A_PRIVATE, b=B_GROUP, e=ENDOWMENT):
    """Grid search over integer contributions 0-10; returns the payoff-maximizing c."""
    grid = np.arange(0, e + 1)
    payoffs = [round_payoff(c, peer_sum, a, b, e) for c in grid]
    return int(grid[np.argmax(payoffs)]), payoffs

# Verify c=0 is the best response regardless of observability regime,
# confirming the Nash prediction in Section 2 does not depend on peer visibility.
for peer_sum in PEER_SCHEDULE:
    best_c, _ = nash_best_response(peer_sum)
    print(f"peer_sum={peer_sum:2d}  best response c*={best_c}")

peer_sum=10  best response c*=0
peer_sum=10  best response c*=0
peer_sum=10  best response c*=0
peer_sum=10  best response c*=0
peer_sum=10  best response c*=0
peer_sum=10  best response c*=0


## Verifying the deployed game (Appendix A test cases)

Two fixed strategies were tested on the live Hugging Face demo and hand-verified against predicted payoffs. This cell reproduces both calculations in Python to confirm the payoff arithmetic is correct.

In [7]:
def run_strategy(contributions, peer_schedule=PEER_SCHEDULE):
    """Run a fixed sequence of own contributions against the peer schedule; return per-round and cumulative payoffs."""
    rounds = []
    cumulative = 0
    for c, peer_sum in zip(contributions, peer_schedule):
        pool = c + peer_sum
        payoff = round_payoff(c, peer_sum)
        cumulative += payoff
        rounds.append({"c": c, "peer_sum": peer_sum, "pool": pool, "payoff": payoff})
    return rounds, cumulative

# Typical: contribute 5 every round
typical_rounds, typical_cumulative = run_strategy([5] * 6)
print("TYPICAL (c=5 every round)")
for r in typical_rounds:
    print(f"  pool={r['pool']:.0f}  payoff=${r['payoff']:.2f}")
print(f"  cumulative payoff: ${typical_cumulative:.2f}\n")

# Boundary: contribute 0 every round
boundary_rounds, boundary_cumulative = run_strategy([0] * 6)
print("BOUNDARY (c=0 every round)")
for r in boundary_rounds:
    print(f"  pool={r['pool']:.0f}  payoff=${r['payoff']:.2f}")
print(f"  cumulative payoff: ${boundary_cumulative:.2f}\n")

assert abs(typical_cumulative - 300.00) < 0.01, "typical cumulative payoff mismatch"
assert abs(boundary_cumulative - 360.00) < 0.01, "boundary cumulative payoff mismatch"
print("Both cumulative totals match the deployed Hugging Face demo exactly.")


TYPICAL (c=5 every round)
  pool=15  payoff=$50.00
  pool=15  payoff=$50.00
  pool=15  payoff=$50.00
  pool=15  payoff=$50.00
  pool=15  payoff=$50.00
  pool=15  payoff=$50.00
  cumulative payoff: $300.00

BOUNDARY (c=0 every round)
  pool=10  payoff=$60.00
  pool=10  payoff=$60.00
  pool=10  payoff=$60.00
  pool=10  payoff=$60.00
  pool=10  payoff=$60.00
  pool=10  payoff=$60.00
  cumulative payoff: $360.00

Both cumulative totals match the deployed Hugging Face demo exactly.


In [8]:
# Multi-schedule sweep: does the boundary-dominates-typical result generalize
# beyond the single fixed schedule used in the deployed demo?
rng = np.random.default_rng(206)
N_TRIALS = 500

def random_peer_schedule(rng, low_range=(5, 15), high_range=(15, 30)):
    low = rng.integers(low_range[0], low_range[1] + 1, size=3)
    high = rng.integers(high_range[0], high_range[1] + 1, size=3)
    return list(low) + list(high)

typical_totals, boundary_totals = [], []
for _ in range(N_TRIALS):
    schedule = random_peer_schedule(rng)
    _, t_total = run_strategy([5] * 6, schedule)
    _, b_total = run_strategy([0] * 6, schedule)
    typical_totals.append(t_total)
    boundary_totals.append(b_total)

typical_totals, boundary_totals = np.array(typical_totals), np.array(boundary_totals)
print(f"Across {N_TRIALS} randomly drawn peer schedules:")
print(f"  boundary (c=0) beats typical (c=5) in {np.mean(boundary_totals > typical_totals)*100:.1f}% of trials")
print(f"  mean boundary payoff: ${boundary_totals.mean():.2f}  mean typical payoff: ${typical_totals.mean():.2f}")
assert np.all(boundary_totals > typical_totals), "boundary should dominate typical in every trial, given a>b"
print("Boundary strictly dominates typical in every trial — confirms the Nash prediction generalizes beyond the single deployed schedule.")

# Adaptive-strategy condition (Section 3's "one modification to the baseline").
# In rounds 4-6, contribution responds to the previous round's visible peer average.
def adaptive_strategy(peer_schedule, low_contribution=0, sensitivity=0.0):
    contributions = []
    for i, peer_sum in enumerate(peer_schedule):
        if i < 3:
            c = low_contribution
        else:
            prev_peer_avg = peer_schedule[i - 1] / 4
            c = max(0, min(10, int(round(sensitivity * prev_peer_avg))))
        contributions.append(c)
    return contributions

print("\nAdaptive-strategy sweep (mechanism demonstration, not a behavioral finding):")
for sensitivity in [0.0, 0.25, 0.5, 1.0]:
    c_seq = adaptive_strategy(PEER_SCHEDULE, sensitivity=sensitivity)
    _, cumulative = run_strategy(c_seq)
    print(f"  sensitivity={sensitivity:.2f}  contributions={c_seq}  cumulative=${cumulative:.2f}")


Across 500 randomly drawn peer schedules:
  boundary (c=0) beats typical (c=5) in 100.0% of trials
  mean boundary payoff: $435.73  mean typical payoff: $375.73
Boundary strictly dominates typical in every trial — confirms the Nash prediction generalizes beyond the single deployed schedule.

Adaptive-strategy sweep (mechanism demonstration, not a behavioral finding):
  sensitivity=0.00  contributions=[0, 0, 0, 0, 0, 0]  cumulative=$360.00
  sensitivity=0.25  contributions=[0, 0, 0, 1, 1, 1]  cumulative=$354.00
  sensitivity=0.50  contributions=[0, 0, 0, 1, 1, 1]  cumulative=$354.00
  sensitivity=1.00  contributions=[0, 0, 0, 2, 2, 2]  cumulative=$348.00


## Your experiment

The multi-schedule sweep and adaptive-strategy mechanism described in Section 3 as "planned, not yet run" are implemented and run above. What remains genuinely untested is whether a *real* human player's contribution actually responds to visible peer behavior the way the adaptive-strategy function assumes — that requires the human-participant test proposed in Section 4, not a synthetic run of a function built to behave that way by construction.

## Sources and scope

Andreoni & Petrie 2004; Crifo, Diaye & Oueghlissi 2017; Nash 1950; Ostrom 1990; Fischbacher, Gächter & Fehr 2001. This notebook adapts the class's example structure (Jia et al., NeurIPS 2025) into an implementation of the author's own PS1 proposal; it is not a reproduction of Jia et al.'s study.